In [1]:
import pandas as pd
import numpy as np

In [ ]:
# ==========================================
# 1. SETUP PATHS
# ==========================================
# Path to your Stata file
dta_path = "/Users/ruben/Desktop/Thesis/TrainingData/PH_DHS/PHHR82DT/PHHR82FL.DTA"

# Output CSV
output_csv = "dhs_wealth.csv"

# ==========================================
# 2. LOAD STATA FILE
# ==========================================
print("Loading PHHR82FL.DTA...")

# We only need two specific columns:
# HV001 = Cluster Number (The ID we match with satellite data)
# HV271 = Wealth Index Factor Score (Continuous number, better for regression)
df = pd.read_stata(dta_path, columns=['hv001', 'hv271'])

print(f"Loaded {len(df)} household records.")

Loading PHHR82FL.DTA...
Loaded 30372 household records.


In [ ]:
# ==========================================
# 3. PROCESS & AGGREGATE
# ==========================================
# The raw HV271 score is usually multiplied by 100,000 in DHS files to avoid decimals.
# We divide by 100,000 to get the "true" score (usually between -2.0 and +3.0)
df['hv271'] = df['hv271'] / 100000.0

# Rename for clarity
df = df.rename(columns={'hv001': 'DHSCLUST', 'hv271': 'Wealth_Index'})

# AGGREGATE: Calculate the Mean Wealth for each Cluster
df_cluster = df.groupby('DHSCLUST')['Wealth_Index'].mean().reset_index()

# ==========================================
# 4. SAVE
# ==========================================
# Ensure Cluster ID is integer
df_cluster['DHSCLUST'] = df_cluster['DHSCLUST'].astype(int)

df_cluster.to_csv(output_csv, index=False)

print("-" * 30)
print(f"SUCCESS: Aggregated wealth data for {len(df_cluster)} clusters.")
print(f"Saved to: {output_csv}")
print(df_cluster.head())

------------------------------
SUCCESS: Aggregated wealth data for 1247 clusters.
Saved to: dhs_wealth.csv
   DHSCLUST  Wealth_Index
0         1     -1.212692
1         2     -0.691787
2         3     -1.304890
3         4     -1.900529
4         5     -1.083710


In [8]:
sample_csv_path = "dynamic_features_per_quarter.csv"
output_csv = "dhs_wealth_sample50.csv"

df_sample = pd.read_csv(sample_csv_path)

# Get the unique IDs (e.g., [1, 4, 12, ...])
target_ids = df_sample['ClusterID'].unique().astype(int)
print(f"Found {len(target_ids)} unique clusters in your sample.")

df_subset = df_cluster[df_cluster['DHSCLUST'].isin(target_ids)].copy()

print(f"Filtered down to {len(df_subset)} households belonging to your {len(target_ids)} clusters.")

df_subset.to_csv(output_csv, index=False)

print("-" * 30)
print(f"SUCCESS: Created targets for {len(df_subset)} clusters.")
print(f"Saved to: {output_csv}")
print(df_subset.head())


Found 50 unique clusters in your sample.
Filtered down to 50 households belonging to your 50 clusters.
------------------------------
SUCCESS: Created targets for 50 clusters.
Saved to: dhs_wealth_sample50.csv
    DHSCLUST  Wealth_Index
30        31     -0.666257
55        56     -0.227968
62        63     -1.303484
66        67      0.321347
73        74      0.393459
